In [5]:
RAW_PATH  = '../data/raw/'
OUT_PATH  = '../outputs/'

In [7]:
import pandas as pd
import numpy as np

# ── 1.1 Carga de archivos ──────────────────────────────────
pacientes  = pd.read_csv(RAW_PATH + '01_pacientes.csv')
atenciones = pd.read_csv(RAW_PATH + '02_atenciones.csv', parse_dates=['fecha_atencion'])
historia   = pd.read_csv(RAW_PATH + '03_historia_clinica_detalle.csv', parse_dates=['fecha_registro'])
prefactura = pd.read_csv(RAW_PATH + '04_prefactura.csv', parse_dates=['fecha_facturacion'])
cruce_gt   = pd.read_csv(RAW_PATH + '05_cruce_validacion.csv')  # ground truth

print("Archivos cargados correctamente")
print(f"Pacientes:       {len(pacientes):,} registros")
print(f"Atenciones:      {len(atenciones):,} registros")
print(f"Historia clínica:{len(historia):,} registros")
print(f"Prefactura:      {len(prefactura):,} registros")
print(f"Cruce (GT):      {len(cruce_gt):,} registros")

Archivos cargados correctamente
Pacientes:       300 registros
Atenciones:      1,200 registros
Historia clínica:3,056 registros
Prefactura:      2,974 registros
Cruce (GT):      3,126 registros


In [8]:
# ── 1.2 Validación de tipos de datos ──────────────────────
print("\nTipos de datos:")
for nombre, df in [('Pacientes', pacientes), ('Atenciones', atenciones),
                   ('Historia', historia), ('Prefactura', prefactura)]:
    print(f"\n{nombre}:")
    print(df.dtypes)


Tipos de datos:

Pacientes:
id_paciente        object
tipo_documento     object
edad                int64
sexo               object
eps                object
tipo_afiliacion    object
ciudad             object
dtype: object

Atenciones:
id_atencion                            object
id_paciente                            object
fecha_atencion                 datetime64[ns]
tipo_atencion                          object
diagnostico_principal_cie10            object
descripcion_diagnostico                object
medico_tratante                        object
sede                                   object
eps                                    object
dtype: object

Historia:
id_detalle                         object
id_atencion                        object
tipo_item                          object
codigo_cups                        object
descripcion                        object
cantidad_realizada                  int64
fecha_registro             datetime64[ns]
soporte_clinico              

In [14]:
# ── 1.3 Revisión de valores nulos ─────────────────────────
print("Valores nulos por dataset:")
for nombre, df in [('Pacientes', pacientes), ('Atenciones', atenciones), ('Historia', historia), ('Prefactura', prefactura), ('Cruce GT', cruce_gt)]:
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]  # solo mostrar columnas con nulos
    if len(nulos) > 0:
        print(f"{nombre}:")
        print(nulos)
    else:
        print(f"{nombre}: sin valores nulos")

Valores nulos por dataset:
Pacientes: sin valores nulos
Atenciones: sin valores nulos
Historia: sin valores nulos
Prefactura: sin valores nulos
Cruce GT:
id_prefactura    152
id_detalle_hc     70
dtype: int64


In [15]:
# ── 1.4 Manejo de nulos en cruce_gt
# Los nulos en id_prefactura corresponden a procedimientos NO_FACTURADO
# (no tienen prefactura porque nunca se cobraron)
# Los nulos en id_detalle_hc corresponden a cobros SIN_SOPORTE_CLINICO
# (no tienen registro en HC porque no se realizaron)

cruce_gt['id_prefactura'] = cruce_gt['id_prefactura'].fillna('SIN_PREFACTURA')
cruce_gt['id_detalle_hc'] = cruce_gt['id_detalle_hc'].fillna('SIN_DETALLE_HC')

print("Nulos manejados en cruce_gt:")
print(f"SIN_PREFACTURA: {(cruce_gt['id_prefactura'] == 'SIN_PREFACTURA').sum()} registros → procedimientos no facturados")
print(f"SIN_DETALLE_HC: {(cruce_gt['id_detalle_hc'] == 'SIN_DETALLE_HC').sum()} registros → cobros sin soporte clínico")

Nulos manejados en cruce_gt:
SIN_PREFACTURA: 152 registros → procedimientos no facturados
SIN_DETALLE_HC: 70 registros → cobros sin soporte clínico


In [16]:
# ── 1.5 Estandarización de columnas clave ─────────────────
# Los códigos CUPS vienen como int en historia y como int en prefactura
# Los convertimos a string para evitar errores de cruce

historia['codigo_cups'] = historia['codigo_cups'].astype(str).str.strip()
prefactura['codigo_cups_facturado'] = prefactura['codigo_cups_facturado'].astype(str).str.strip()

print("Códigos CUPS estandarizados a string")

Códigos CUPS estandarizados a string


In [17]:
# ── 1.6 Estadísticas descriptivas

print(f"\nAtenciones por tipo:")
print(atenciones['tipo_atencion'].value_counts())

print(f"\nProcedimientos en HC por tipo:")
print(historia['tipo_item'].value_counts())

print(f"\nValor total en prefactura: ${prefactura['valor_total'].sum():,.0f}")
print(f"Valor promedio por ítem: ${prefactura['valor_total'].mean():,.0f}")
print(f"Valor máximo facturado: ${prefactura['valor_total'].max():,.0f}")

print(f"\nDistribución de alertas esperadas (ground truth):")
print(cruce_gt['tipo_alerta'].value_counts())


Atenciones por tipo:
tipo_atencion
Urgencias          421
Ambulatoria        407
Hospitalizacion    372
Name: count, dtype: int64

Procedimientos en HC por tipo:
tipo_item
consulta       1492
examen         1108
tratamiento     456
Name: count, dtype: int64

Valor total en prefactura: $891,186,000
Valor promedio por ítem: $299,659
Valor máximo facturado: $5,200,000

Distribución de alertas esperadas (ground truth):
tipo_alerta
CONSISTENTE                   2477
SIN_SOPORTE_CLINICO            157
NO_FACTURADO                   152
DIAGNOSTICO_NO_RELACIONADO     152
CODIGO_NO_COINCIDE             120
CANTIDAD_DISCORDANTE            68
Name: count, dtype: int64
